In [ ]:
#!/usr/bin/env python3
import os
import time
import json
import requests
import math
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

API_KEY = ""

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

CSV_PATH = "BSMDD_stratified_200.csv"

MODEL = "openai/gpt-5-mini"  # change

FEW_SHOT_EXAMPLES = [
    {
        "text": "আত্মহত্যার করতাম দুধারের তরবারির কষ্টের অবসান ঘটাচ্ছে ব্যথা দিচ্ছে চারপাশে আটকে রাখছে অতীতে নিজেকে কেটে ফেলেছি বাঁচতে চাই না।",
        "label": "1"
    },
    {
        "text": "জন্মদিন বার্ষিকী অনুসন্ধান উদ্ধারকারী নিখোঁজ ভাইকে সন্ধান ছেড়ে মাস অবশেষে লাশ খুঁজে পায় জন্মদিনকে ঘৃণা জন্মিতাম অস্তিত্ব থাকত না।",
        "label": "1"
    },
    {
        "text": "ছোট ভাইবোনরা সবচেয়ে খারাপ বোন অনলাইন টিভি বন্ধ বললো স্কুল মাকে কর্মস্থলে ভিডিও গেম খেলতে স্থান করছি দুঃখিত পাশে খেলতে।",
        "label": "0"
    },
    {
        "text": "অনলাইনে বন্ধু তৈরি কঠিন এমনকি বন্ধু তাদেরও সবসময় অনলাইন বন্ধু কখনোই তৈরি পারিনি ব্যক্তিগতভাবে বন্ধুত্ব এতটা খারাপ একাকী বয়সী মেয়ে।",
        "label": "0"
    },
]

SYSTEM_PROMPT = """Classify the given Bangla social media text into following categories:
1 = Depressive (hopelessness, suicidal, self-harm, clear depression).
0 = Non-depressive (stress/anger alone doesn't equal depression).
Output EXACTLY '0' or '1'. No explanations. Nothing else. only "0" and "1"."""

MAX_RETRIES = 5
BASE_BACKOFF = 2.0
REQUEST_TIMEOUT = 30

SITE_URL = "https://colab.research.google.com"
SITE_NAME = "Bangla Benchmark"


def build_headers():
    if not API_KEY:
        raise RuntimeError("API_KEY not set")
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": SITE_URL,
        "X-Title": SITE_NAME,
    }


def build_few_shot_messages(text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for example in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": f"Text: {example['text']}\nLabel:"})
        messages.append({"role": "assistant", "content": example["label"]})
    messages.append({"role": "user", "content": f"Text: {text}\nLabel:"})
    return messages


def parse_label(raw_text):
    if raw_text is None:
        return None
    for ch in raw_text.strip():
        if ch == "0":
            return 0
        if ch == "1":
            return 1
    return None


def log_reasoning(data, choice):
    usage = data.get("usage", {})
    reasoning_tokens = usage.get("completion_tokens_details", {}).get("reasoning_tokens", None)
    has_reasoning_field = "reasoning" in choice.get("message", {})

    if reasoning_tokens is not None:
        if reasoning_tokens > 0:
            status = f"ON  ({reasoning_tokens} reasoning tokens)"
        else:
            status = "OFF (0 reasoning tokens — effort:none confirmed)"
    elif has_reasoning_field:
        preview = str(choice["message"]["reasoning"])[:60]
        status = f"ON  (field present: {preview})"
    else:
        status = "OFF (no reasoning field or tokens reported)"

    print(f"  [reasoning] {status}")


def call_model(model, text, headers):
    messages = build_few_shot_messages(text)
    # payload = { //for gemini 3.6 flash
    #   "model": model,
    #   "messages": messages,
    #   "max_tokens": 5,
    #   "temperature": 0,
    #   "thinking_level": "minimal"
    # }

    payload = {
      "model": "gpt-5-mini",
      "messages": messages,
      "max_tokens": 100,
      "temperature": 0,
      "reasoning": {
         "effort": "minimal"
      }
    }

    start = time.monotonic()

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                OPENROUTER_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT
            )
        except requests.RequestException as e:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{e}", time.monotonic() - start
            time.sleep(BASE_BACKOFF * attempt)
            continue

        if resp.status_code == 429:
            retry_after = resp.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else BASE_BACKOFF * attempt
            time.sleep(wait)
            continue

        if resp.status_code >= 500:
            time.sleep(BASE_BACKOFF * attempt)
            continue

        if resp.status_code != 200:
            return None, f"http_error:{resp.status_code}:{resp.text[:200]}", time.monotonic() - start

        try:
            data = resp.json()
            choice = data["choices"][0]
            content = choice["message"]["content"]
            log_reasoning(data, choice)
        except (KeyError, IndexError, json.JSONDecodeError) as e:
            return None, f"parse_error:{e}", time.monotonic() - start

        return content, None, time.monotonic() - start

    return None, "max_retries_exceeded", time.monotonic() - start


def checkpoint_path(out_dir, model):
    col_name = model.replace("/", "__")
    return os.path.join(out_dir, f"checkpoint_fewshot_{col_name}.csv")


def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    if os.path.exists(path):
        print(f"[checkpoint] Resuming from {path}")
        return pd.read_csv(path)
    return None


def save_checkpoint(out_dir, model, rows):
    path = checkpoint_path(out_dir, model)
    pd.DataFrame(rows).to_csv(path, index=False)


def run_model_on_dataset(model, df, headers, out_dir, checkpoint_every=20):
    existing = load_checkpoint(out_dir, model)
    rows = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows)

    if start_idx >= len(df):
        print(f"[checkpoint] Already complete — {start_idx} rows done.")
        return rows

    texts = df["text"].tolist()
    labels = df["label"].tolist()

    pbar = tqdm(
        range(start_idx, len(df)),
        desc=model,
        unit="row",
        initial=start_idx,
        total=len(df),
    )

    for i in pbar:
        text = texts[i]
        label = labels[i]

        print(f"\n[row {i}] Calling model...")
        raw, err, latency = call_model(model, text, headers)
        pred = parse_label(raw)

        if pred is None:
            print(f"  [retry] pred=None on first call, retrying...")
            raw, err, latency2 = call_model(model, text, headers)
            pred = parse_label(raw)
            latency += latency2

        correct = (pred == label) if pred is not None else False
        print(f"  [result] raw='{raw}' → pred={pred} | label={label} | correct={correct}")

        rows.append({
            "text": text,
            "label": label,
            "prediction": pred,
            "correct": correct,
            "latency": latency,
            "raw_response": raw,
            "error": err,
        })

        if (i - start_idx + 1) % checkpoint_every == 0:
            save_checkpoint(out_dir, model, rows)
            print(f"  [checkpoint] Saved at row {i}")

    save_checkpoint(out_dir, model, rows)
    return rows


def compute_metrics(rows):
    import math
    valid = [r for r in rows if r["prediction"] is not None and not (isinstance(r["prediction"], float) and math.isnan(r["prediction"]))]
    if not valid:
        return {
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1": None,
            "confusion_matrix": None,
            "valid_predictions": 0,
            "total": len(rows),
            "avg_latency": None,
        }

    yt = [int(r["label"]) for r in valid]
    yp = [int(r["prediction"]) for r in valid]
    latencies = [r["latency"] for r in rows if r["latency"] is not None]

    return {
        "accuracy": accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "confusion_matrix": confusion_matrix(yt, yp, labels=[0, 1]).tolist(),
        "valid_predictions": len(valid),
        "total": len(rows),
        "avg_latency": sum(latencies) / len(latencies) if latencies else None,
    }


def main():
    NUM_SAMPLES = 100      # set to int to limit rows
    OUTPUT_DIR = "/content"
    CHECKPOINT_EVERY = 10

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)

    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError("CSV must contain 'text' and 'label' columns")

    df["label"] = df["label"].astype(int)

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"[config] Model      : {MODEL}")
    print(f"[config] Prompting  : 4-shot")
    print(f"[config] Rows       : {len(df)}")
    print(f"[config] Checkpoint : every {CHECKPOINT_EVERY} rows")
    print(f"[config] Reasoning  : effort=none (watch per-row log below)\n")

    headers = build_headers()

    rows = run_model_on_dataset(MODEL, df, headers, OUTPUT_DIR, CHECKPOINT_EVERY)

    pred_rows = [
        {
            "text": r["text"],
            "label": r["label"],
            "prediction": r["prediction"],
            "correct": r["correct"],
            "latency": r["latency"],
            "raw_response": r["raw_response"],
            "error": r.get("error"),
        }
        for r in rows
    ]

    metrics = compute_metrics(rows)

    results_row = {
        "model": MODEL,
        "prompting": "4-shot",
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "confusion_matrix": json.dumps(metrics["confusion_matrix"]),
        "valid_predictions": metrics["valid_predictions"],
        "total": metrics["total"],
        "avg_latency": metrics["avg_latency"],
    }

    model_slug = MODEL.replace("/", "_")
    pred_path = os.path.join(OUTPUT_DIR, f"{model_slug}_{NUM_SAMPLES}_predictions_fewshot.csv")
    pd.DataFrame(pred_rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

    results_path = os.path.join(OUTPUT_DIR, f"{model_slug}_{NUM_SAMPLES}_benchmark_results_fewshot.csv")
    pd.DataFrame([results_row]).to_csv(results_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 60)
    print(pd.DataFrame([results_row]).to_string(index=False))


if __name__ == "__main__":
    main()

[config] Model      : openai/gpt-5-mini
[config] Prompting  : 4-shot
[config] Rows       : 100
[config] Checkpoint : every 10 rows
[config] Reasoning  : effort=none (watch per-row log below)



openai/gpt-5-mini:   0%|          | 0/100 [00:00<?, ?row/s]


[row 0] Calling model...


openai/gpt-5-mini:   1%|          | 1/100 [00:00<01:33,  1.06row/s]

  [reasoning] OFF (0 reasoning tokens — effort:none confirmed)
  [result] raw='0' → pred=0 | label=0 | correct=True

[row 1] Calling model...


...

  [reasoning] OFF (0 reasoning tokens — effort:none confirmed)
  [result] raw='0' → pred=0 | label=0 | correct=True

[row 98] Calling model...


openai/gpt-5-mini:  99%|█████████▉| 99/100 [01:46<00:00,  1.04row/s]

  [reasoning] OFF (0 reasoning tokens — effort:none confirmed)
  [result] raw='0' → pred=0 | label=0 | correct=True

[row 99] Calling model...


openai/gpt-5-mini: 100%|██████████| 100/100 [01:47<00:00,  1.07s/row]

  [reasoning] OFF (0 reasoning tokens — effort:none confirmed)
  [result] raw='0' → pred=0 | label=0 | correct=True
  [checkpoint] Saved at row 99

            model prompting  accuracy  precision  recall       f1   confusion_matrix  valid_predictions  total  avg_latency
openai/gpt-5-mini    4-shot      0.87   0.893617    0.84 0.865979 [[45, 5], [8, 42]]                100    100     1.070737
